# **Orchestrator–Worker Pattern**

The **Orchestrator** breaks a complex query into sub-tasks.
A **Worker** executes all sub-tasks concurrently using `ThreadPoolExecutor`.
A **Collector** summarises the parallel results.

Flow: `START → orchestrator → worker → collector → END`

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ.get('GROQ_API_KEY'):
    print("Groq API Key is set.")
else:
    raise ValueError("GROQ_API_KEY not found.")

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0)

### **Pydantic LLM Schema**
The orchestrator needs to return a structured list of tasks, not free text.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, TypedDict

class llm_schema(BaseModel):
    tasks: List[str] = Field(..., description="A list of task prompts for the worker to complete.")

# Wrap the LLM so it always returns an llm_schema object
llm_with_schema = llm.with_structured_output(llm_schema)

### **Graph Schema**

In [ ]:
class graph_schema(TypedDict):
    tasks:   List[str]   # sub-tasks generated by the orchestrator
    query:   str         # the user's original complex question
    results: List[str]   # one result string per task, from the worker
    summary: str         # the collector's summary of all results

### **Orchestrator Node**
Breaks the user query into discrete sub-task prompts.

In [ ]:
def orchestrator_node(state: graph_schema) -> graph_schema:

    user_query = state['query']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an orchestrator that breaks down a user query into tasks for the worker."),
        ("user",   f"User query: {user_query}. Generate one prompt per task for the worker. Return the tasks as a list.")
    ])

    chain    = prompt | llm_with_schema
    response = chain.invoke({"query": user_query})

    # response.tasks is a List[str] — one prompt per sub-task
    state['tasks'] = response.tasks
    return state

### **Worker Node**
Executes all tasks **concurrently** using threads to minimise total wait time.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def execute(query: str) -> str:
    """Runs a single task by calling the LLM. Called from multiple threads in parallel."""
    response = llm.invoke(f"Please execute this task: {query}")
    return response.content


def worker_node(state: graph_schema) -> graph_schema:

    tasks = state['tasks']

    # ThreadPoolExecutor runs each task in its own thread simultaneously
    with ThreadPoolExecutor(max_workers=len(tasks)) as executor:
        results = list(executor.map(execute, tasks))   # blocks until all tasks are done

    state['results'] = results
    return state

### **Collector Node**
Summarises all the worker's results into a single cohesive answer.

In [ ]:
def collector_node(state: graph_schema) -> graph_schema:

    results = state['results']

    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a collector that summarises results from a worker."),
        ("user",   f"Here are the results from the worker: {results}. Summarise them concisely.")
    ])

    chain   = prompt | llm
    summary = chain.invoke({"results": results})

    state['summary'] = summary.content
    return state

### **Build and Run the Graph**

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image

graph = StateGraph(graph_schema)

graph.add_node("orchestrator_node", orchestrator_node)
graph.add_node("worker_node",       worker_node)
graph.add_node("collector_node",    collector_node)

# Linear flow — each stage feeds into the next
graph.add_edge(START, "orchestrator_node")
graph.add_edge("orchestrator_node", "worker_node")
graph.add_edge("worker_node",       "collector_node")
graph.add_edge("collector_node",    END)

complex_graph = graph.compile()
Image(complex_graph.get_graph().draw_mermaid_png())

In [ ]:
result = complex_graph.invoke({
    "query":   "What is the capital of France and population of Paris? & What is the capital of Germany and population of Berlin?",
    "tasks":   [],
    "results": [],
    "summary": ""
})
print(result['summary'])

In [ ]:
# stream() shows updates node by node — useful for watching progress
for chunk in complex_graph.stream(
    {
        "query":   "What is the capital of France and population of Paris? & What is the capital of Germany and population of Berlin?",
        "tasks":   [],
        "results": [],
        "summary": ""
    },
    stream_mode="updates"
):
    print(chunk)